# PH-SHOWOA · Python Full-GPU Tensorized SA-RCRS-GRASP Solver Benchmark

**Mục tiêu**: Chạy thuật toán PH-SHOWOA Python Full-GPU Tensorized (`src_python_gpu_SA_RCRS_GRASP`) tăng tốc PyTorch GPU (CUDA) với phương pháp khởi tạo kết hợp **SA-RCRS-GRASP** trên 15 bộ dữ liệu VRPSPDTW chuẩn của Wang & Chen.

**Thông số thử nghiệm**:
- `Compute Backend` = `cuda` (PyTorch 3D Tensorized CUDA GPU Evaluation & Acceleration)
- `Init` = `sa_rcrs_grasp` (Tự động kèm RCRS-GRASP RCL + 25 vòng SA Post-refinement)
- `Popsize` = 36 (6x6 Perfect Square Island Grid)
- `Max-iteration` = 1000
- `Runs` = 30

**GPU khuyên dùng trên Kaggle**: **NVIDIA Tesla T4** / **NVIDIA P100** / **NVIDIA L4**

## Cell 1 – Clone hoặc Cập nhật Repo từ GitHub

In [ ]:
import os
if os.path.exists("ph-showoa"):
    !cd ph-showoa && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/Welkie/ph-showoa.git


## Cell 2 – Kiểm tra GPU & Môi trường PyTorch CUDA

In [ ]:
!nvidia-smi
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))

## Cell 3 – Batch runner Python Full-GPU Tensorized SA-RCRS-GRASP (15 bộ dữ liệu)

In [ ]:
import os, glob, subprocess, re, sys, csv, time as T
from collections import deque
import pandas as pd

# ── Cấu hình 15 bộ dữ liệu ───────────────────────────────────────────────────
DATASETS = [
    "rcdp1001", "rcdp5001", "rcdp5007", "rcdp5004", "rcdp101",
    "cdp103",   "rcdp205",  "rdp210",   "rcdp207",  "rcdp202",
    "rdp103",   "cdp104",   "cdp102",   "rdp203",   "rcdp104"
]

CWD             = "ph-showoa" if os.path.exists("ph-showoa") else "."
DATASET_DIR     = "/kaggle/input/datasets/keith1101/ph-showoa/Wang_Chen"
OUTPUT_CSV      = "/kaggle/working/summary_python_full_gpu_tensorized.csv"
LOG_DIR         = "/kaggle/working/logs"
RUNS            = 30
MAX_ITER        = 1000
POP_SIZE        = 36  # Phải là số chính phương (6x6)
INIT_MODE       = "sa_rcrs_grasp"
COMPUTE_BACKEND = "cuda"
WORKERS         = 1
RESUME          = True  # Tự động nạp và bỏ qua các dataset đã giải thành công trước đó
MAX_TIMEOUT_S   = None  # Khong gioi han thoi gian 

TAIL_LINES      = 10
os.makedirs(LOG_DIR, exist_ok=True)

# ── Helpers ───────────────────────────────────────────────────────────────────
def find_file(name):
    # 1. Local repo dataset directory
    p_repo = os.path.join(CWD, "dataset", f"explicit_{name}.vrpsdptw")
    if os.path.exists(p_repo):
        return p_repo
    
    # 2. Kaggle input directory
    p_input = os.path.join(DATASET_DIR, f"explicit_{name}.vrpsdptw")
    if os.path.exists(p_input):
        return p_input
    
    # 3. Recursive glob search
    found = glob.glob(f"/kaggle/input/**/explicit_{name}.vrpsdptw", recursive=True)
    if not found:
        found = glob.glob(f"/kaggle/input/**/[eE][xX][pP][lL][iI][cC][iI][tT]_{name}.vrpsdptw", recursive=True)
    if not found:
        found = glob.glob(f"**/explicit_{name}.vrpsdptw", recursive=True)
    return found[0] if found else None

def parse_output(text):
    runs_m  = re.search(r"Total (\d+) runs, total consumed (\d+) sec", text)
    nv_m    = re.search(r"Vehicle count:\s*(\d+)", text)
    cost_m  = re.search(r"Total cost:\s*([\d.]+)", text)
    if runs_m and nv_m and cost_m:
        tr   = int(runs_m.group(1))
        ts   = float(runs_m.group(2))
        nv   = int(nv_m.group(1))
        cost = float(cost_m.group(1))
        td   = cost - 2000.0 * nv
        return {"best_NV": nv, "best_TD": f"{td:.4f}",
                "total_cost": f"{cost:.4f}",
                "avg_time_s": f"{ts/tr:.2f}" if tr else "N/A",
                "total_runs": tr, "Status": "Success"}
    return {"best_NV":"N/A","best_TD":"N/A","total_cost":"N/A",
            "avg_time_s":"N/A","total_runs":0,"Status":"Parse Error"}

def save_summary_csv(res_list, csv_path):
    if not res_list:
        return
    cols = ["Dataset", "best_NV", "best_TD", "total_cost", "avg_time_s", "wall_time", "total_runs", "Status"]
    df = pd.DataFrame(res_list)
    for c in cols:
        if c not in df.columns:
            df[c] = "N/A"
    df = df[cols]
    df.columns = ["Dataset", "Best NV", "Best TD", "Total Cost", "Avg/Run", "Wall Time", "Runs", "Status"]
    df.to_csv(csv_path, index=False)

# ── Load existing results if RESUME is enabled ─────────────────────────────────
results = []
completed_names = set()

if RESUME and os.path.exists(OUTPUT_CSV):
    try:
        prev_df = pd.read_csv(OUTPUT_CSV)
        for _, row in prev_df.iterrows():
            st = str(row.get("Status", "")).strip()
            if st.lower() == "success":
                r_entry = {
                    "Dataset": row.get("Dataset"),
                    "best_NV": row.get("Best NV", row.get("best_NV")),
                    "best_TD": row.get("Best TD", row.get("best_TD")),
                    "total_cost": row.get("Total Cost", row.get("total_cost")),
                    "avg_time_s": row.get("Avg/Run", row.get("avg_time_s")),
                    "wall_time": row.get("Wall Time", row.get("wall_time")),
                    "total_runs": row.get("Runs", row.get("total_runs")),
                    "Status": "Success"
                }
                results.append(r_entry)
                completed_names.add(str(row.get("Dataset")).strip())
        if completed_names:
            print(f"[RESUME] Đã nạp {len(completed_names)} bộ dữ liệu đã giải thành công trước đó:")
            print(f"         {sorted(list(completed_names))}\n")
    except Exception as e:
        print(f"[RESUME WARNING] Không đọc được file cũ ({e}), bắt đầu lại từ đầu.\n")

# ── Batch loop ────────────────────────────────────────────────────────────────
print("="*70)
print(f"  PH-SHOWOA PYTHON FULL-GPU TENSORIZED  |  runs={RUNS}  iter={MAX_ITER}  pop={POP_SIZE}  backend={COMPUTE_BACKEND}")
print("="*70)

for name in DATASETS:
    if name in completed_names:
        print(f"\n{'─'*60}")
        print(f"  ▶  {name}  [ĐÃ HOÀN THÀNH → BỎ QUA]")
        print(f"{'─'*60}")
        continue

    print(f"\n{'─'*60}")
    print(f"  ▶  {name}")
    print(f"{'─'*60}")

    fp = find_file(name)
    if fp is None:
        print(f"  [SKIP] File not found for: {name}")
        skip_res = {"Dataset": name, "best_NV": "N/A", "best_TD": "N/A",
                    "total_cost": "N/A", "avg_time_s": "N/A",
                    "wall_time": "N/A", "total_runs": 0, "Status": "File Not Found"}
        results.append(skip_res)
        save_summary_csv(results, OUTPUT_CSV)
        continue

    print(f"  Dataset file: {fp}")

    # Khởi tạo Python Full-GPU Tensorized SA-RCRS-GRASP với -u (unbuffered)
    cmd = [
        sys.executable, "-u", "-m", "src_python_gpu_SA_RCRS_GRASP.main",
        "--problem", fp,
        "--compute_backend", COMPUTE_BACKEND,
        "--init", INIT_MODE,
        "--grasp_alpha_lo", "0.10",
        "--grasp_alpha_hi", "0.40",
        "--sa_iterations", "25",
        "--runs",     str(RUNS),
        "--max_iter", str(MAX_ITER),
        "--pop_size", str(POP_SIZE),
        "--workers",  str(WORKERS)
    ]

    t0 = T.time()
    proc = subprocess.Popen(
        cmd, cwd=CWD,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )

    all_lines  = []
    tail_buf   = deque(maxlen=TAIL_LINES)
    last_print = T.time()

    try:
        for line in proc.stdout:
            all_lines.append(line)
            clean_line = line.rstrip()
            tail_buf.append(clean_line)

            now = T.time()
            is_milestone = any(k in clean_line for k in [
                "Pure GPU Multi-Island", "Initialization done", "Best solution found", "Time to surpass BKS"
            ])

            if is_milestone:
                elapsed = now - t0
                print(f"  [{elapsed:.0f}s] {clean_line[:90]}", flush=True)
                last_print = now
            elif now - last_print >= 20:
                elapsed = now - t0
                print(f"  [{elapsed:.0f}s] running... last: {tail_buf[-1][:80]}", flush=True)
                last_print = now

            if MAX_TIMEOUT_S and (now - t0 > MAX_TIMEOUT_S):
                print(f"  [TIMEOUT] Quá thời gian tối đa {MAX_TIMEOUT_S}s, dừng dataset {name}...")
                proc.kill()
                break

        proc.wait()
    except Exception as exc:
        print(f"  [ERROR] Ngoại lệ tiến trình: {exc}")
        proc.kill()

    wall = T.time() - t0

    # Lưu log chi tiết dataset
    log_file = os.path.join(LOG_DIR, f"{name}.log")
    try:
        with open(log_file, "w", encoding="utf-8") as f:
            f.writelines(all_lines)
    except Exception:
        pass

    print(f"  Wall time: {wall:.1f}s")
    print("  --- Last output ---")
    for ln in tail_buf:
        print(" ", ln)

    full_output = "".join(all_lines)
    res = parse_output(full_output)
    res["Dataset"]   = name
    res["wall_time"] = f"{wall:.1f}s"
    results.append(res)
    completed_names.add(name)

    # Lưu kết quả lũy kế ngay lập tức sau mỗi dataset
    save_summary_csv(results, OUTPUT_CSV)
    print(f"  ✓ best_NV={res['best_NV']}  best_TD={res['best_TD']}  status={res['Status']}")
    print(f"  [SAVED] Bảng tổng kết đã cập nhật → {OUTPUT_CSV}")

print(f"\n{'='*70}")
print("  Hoàn tất toàn bộ 15 bộ dữ liệu.")
print('='*70)


## Cell 4 – Bảng tổng hợp kết quả & xuất CSV

In [ ]:
import os
import pandas as pd

OUTPUT_CSV = "/kaggle/working/summary_python_full_gpu_tensorized.csv"

if os.path.exists(OUTPUT_CSV):
    df = pd.read_csv(OUTPUT_CSV)
    print("BẢNG TỔNG HỢP KẾT QUẢ BENCHMARK (PH-SHOWOA PYTHON FULL-GPU TENSORIZED)\n")
    print(df.to_string(index=False))
    print(f"\n[OK] Đã lưu bảng kết quả hoàn chỉnh tại: {OUTPUT_CSV}")
else:
    print(f"Chưa tìm thấy file kết quả {OUTPUT_CSV}.")
